# 03 - Dense GRPO Reinforcement Learning for 3B and 7B SLM
## Text-to-SQL SLM Research Architecture (Phase 2: Policy Optimization)

This notebook executes **Group Relative Policy Optimization (GRPO)** with Dense Multi-Head Rewards:
1. **Dynamic Hardware Configuration Switch**: Automatically configures:
   - **Free T4**: QLoRA 4-bit quantization, micro-batch 1, $K=8$ rollouts.
   - **Pro L4**: BF16 precision, micro-batch 2, $K=8$ rollouts.
   - **Pro+ A100**: BF16 precision, micro-batch 4, $K=16$ rollouts.
2. **Dual-Model Support**: Switch seamlessly between **3B (FINER-SQL-3B)** and **7B (Qwen2.5-Coder-7B)**.
3. **Google Drive Checkpoint Sync**: Automated Drive mount and `--resume_from_checkpoint latest` recovery.
4. **Dense 4-Head Rewards**:
   - $R_{\text{format}}$: `<think>...</think><answer>...</answer>` syntax validation.
   - $R_{\text{exec}}$: Direct execution on SQLite database (0.0, 1.0, 2.0).
   - $R_{\text{mem}}$: Semantic reasoning alignment with Reasoning Bank.
   - $R_{\text{atomic}}$: AST clause matching (SELECT, FROM, WHERE, GROUP BY, ORDER BY).
5. **Group Advantage Normalization**: Eliminates advantage collapse via $A_j = \frac{R_j - \bar{R}}{\sigma + \epsilon}$.


In [ ]:
# Step 1: Workspace Path Setup, Hardware Detection & Hardware Tier Switch
import os
import sys

WORKSPACE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists("../src") else os.path.abspath(".")
if WORKSPACE_DIR not in sys.path:
    sys.path.insert(0, WORKSPACE_DIR)

import json
import math
import subprocess

try:
    import torch
    gpu_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if gpu_available else "CPU (Fallback)"
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if gpu_available else 0.0
except ImportError:
    torch = None
    gpu_available = False
    device_name = "CPU (Fallback)"
    total_vram_gb = 0.0

print("=== Colab Hardware Detection & Tier Switch ===")
print(f"Detected Hardware: {device_name} (VRAM: {total_vram_gb:.2f} GB)")

# Dynamic Hardware Tier Selection
if total_vram_gb >= 40.0:
    HARDWARE_TIER = "Pro+ A100"
    MICRO_BATCH_SIZE = 4
    NUM_ROLLOUTS = 16  # K=16 rollouts
    PRECISION = "bf16"
    QUANTIZATION = None
    GRADIENT_ACCUMULATION = 2
elif total_vram_gb >= 20.0:
    HARDWARE_TIER = "Pro L4"
    MICRO_BATCH_SIZE = 2
    NUM_ROLLOUTS = 8   # K=8 rollouts
    PRECISION = "bf16"
    QUANTIZATION = None
    GRADIENT_ACCUMULATION = 4
else:
    HARDWARE_TIER = "Free T4"
    MICRO_BATCH_SIZE = 1
    NUM_ROLLOUTS = 8   # K=8 rollouts
    PRECISION = "fp16"
    QUANTIZATION = "4bit"  # QLoRA 4-bit
    GRADIENT_ACCUMULATION = 8

print(f"\nActive Hardware Configuration Tier: [{HARDWARE_TIER}]")
print(f"  - Micro-batch size:      {MICRO_BATCH_SIZE}")
print(f"  - Rollouts per query (K):{NUM_ROLLOUTS}")
print(f"  - Precision:             {PRECISION}")
print(f"  - Quantization:          {QUANTIZATION}")

In [ ]:
# Step 2: Model Architecture Selection (Supports 3B and 7B)
# Choose model scale: "3B" (FINER-SQL-3B) or "7B" (Qwen2.5-Coder-7B)
MODEL_SCALE = "3B"

MODEL_CONFIGS = {
    "3B": {
        "name": "FINER-SQL-3B",
        "model_id": "meta-llama/Llama-3.2-3B-Instruct",
        "lora_r": 64,
        "lora_alpha": 16,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "max_seq_len": 1024 if HARDWARE_TIER == "Free T4" else 2048
    },
    "7B": {
        "name": "Qwen2.5-Coder-7B",
        "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
        "lora_r": 128,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "max_seq_len": 2048
    }
}

active_model_cfg = MODEL_CONFIGS[MODEL_SCALE]
print(f"Selected Model Scale:    {MODEL_SCALE} ({active_model_cfg['name']})")
print(f"Base Model ID:           {active_model_cfg['model_id']}")
print(f"LoRA Rank / Alpha:       r={active_model_cfg['lora_r']}, alpha={active_model_cfg['lora_alpha']}")
print(f"Target Modules:          {active_model_cfg['target_modules']}")
print(f"Max Sequence Length:     {active_model_cfg['max_seq_len']}")

In [ ]:
# Step 3: Google Drive Automated Mounting & Checkpoint Setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = f"/content/drive/MyDrive/Text2SQL_Colab/checkpoints/grpo_{MODEL_SCALE.lower()}"
    print("Google Drive mounted at /content/drive")
except (ImportError, Exception):
    DRIVE_CHECKPOINT_DIR = os.path.abspath(f"./colab_drive_mock/checkpoints/grpo_{MODEL_SCALE.lower()}")
    print(f"Using local Drive path: {DRIVE_CHECKPOINT_DIR}")

LOCAL_CHECKPOINT_DIR = f"/content/checkpoints/grpo_{MODEL_SCALE.lower()}" if os.path.exists("/content") else os.path.abspath(f"./checkpoints/grpo_{MODEL_SCALE.lower()}")

os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_CHECKPOINT_DIR, exist_ok=True)

print(f"Drive Checkpoint Path: {DRIVE_CHECKPOINT_DIR}")
print(f"Local Checkpoint Path: {LOCAL_CHECKPOINT_DIR}")


In [ ]:
# Step 4: Session Recovery (--resume_from_checkpoint latest)
from src.training.callbacks import find_latest_checkpoint, get_resume_checkpoint

# Supports CLI argument: --resume_from_checkpoint latest
RESUME_FLAG = "latest"

resume_checkpoint_path = get_resume_checkpoint(
    resume_flag=RESUME_FLAG,
    local_dir=LOCAL_CHECKPOINT_DIR,
    drive_dir=DRIVE_CHECKPOINT_DIR
)

if resume_checkpoint_path:
    print(f"[RECOVERY] Found checkpoint to resume: {resume_checkpoint_path}")
else:
    print("[NEW SESSION] No GRPO checkpoint found. Initializing policy from SFT weights.")


In [ ]:
# Step 5: Dense GRPO 4-Head Reward Functions Implementation
from src.training.train_grpo_generator import (
    compute_format_reward,
    compute_exec_reward,
    compute_mem_reward,
    compute_atomic_reward,
    compute_dense_rewards
)

print("Testing Dense GRPO 4-Reward Heads on sample trajectory...")
sample_response = (
    "<think>\n"
    "1. Table accounts contains client accounts.\n"
    "2. Filter where district_id equals 1.\n"
    "</think>\n"
    "<answer>\n"
    "SELECT account_id FROM accounts WHERE district_id = 1;\n"
    "</answer>"
)
gold_sql = "SELECT account_id FROM accounts WHERE district_id = 1;"
db_file = "data/mini/financial_mini.sqlite" if os.path.exists("data/mini/financial_mini.sqlite") else "../data/mini/financial_mini.sqlite"

r_fmt = compute_format_reward(sample_response)
r_exec = compute_exec_reward(sample_response, gold_sql=gold_sql, db_path=db_file)
r_mem = compute_mem_reward(sample_response, reasoning_bank=["Table accounts contains client accounts. Filter district_id."])
r_atom = compute_atomic_reward(sample_response, gold_sql=gold_sql)

print(f"R_format (Syntax Tags):  {r_fmt:.2f}")
print(f"R_exec (SQLite Match):   {r_exec:.2f}")
print(f"R_mem (Reasoning Sim):   {r_mem:.2f}")
print(f"R_atomic (Clause Match): {r_atom:.2f}")
print(f"Dense Composite Reward:  {r_fmt + r_exec + r_mem + r_atom:.2f}")


In [ ]:
# Step 6: Group Advantage Normalization (Preventing Advantage Collapse)
from src.training.train_grpo_generator import normalize_group_advantages, compute_grpo_loss

print(f"Verifying Advantage Normalization for K={NUM_ROLLOUTS} rollouts...")
# Simulate K rollouts where one succeeds and others partially fail
simulated_rewards = [4.5, 2.0, 1.5, 1.0, 0.5, 0.5, 0.0, 0.0] if NUM_ROLLOUTS == 8 else [4.5, 3.0, 2.5, 2.0, 1.8, 1.5, 1.2, 1.0, 0.8, 0.5, 0.5, 0.2, 0.0, 0.0, 0.0, 0.0]

advantages = normalize_group_advantages(simulated_rewards, eps=1e-6)

print(f"Simulated Rewards:     {simulated_rewards}")
print(f"Normalized Advantages: {[round(a, 3) for a in advantages]}")
print(f"Mean Advantage:        {sum(advantages)/len(advantages):.6f} (Target: ~0.0)")
assert math.isclose(sum(advantages)/len(advantages), 0.0, abs_tol=1e-4)
print("[SUCCESS] Advantage Collapse successfully prevented: gradient variance preserved.")

In [ ]:
# Step 7: DriveCheckpointCallback Integration
from src.training.callbacks import DriveCheckpointCallback

drive_callback = DriveCheckpointCallback(
    drive_checkpoint_dir=DRIVE_CHECKPOINT_DIR,
    max_drive_checkpoints=5,
    sync_every_n_steps=50
)

print(f"Attached DriveCheckpointCallback saving to: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# Step 8: Execute Dense GRPO Training
from src.training.train_grpo_generator import run_grpo_training

DATA_BIRD = "data/mini/mini_bird.jsonl" if os.path.exists("data/mini/mini_bird.jsonl") else "../data/mini/mini_bird.jsonl"
DB_DIR = os.path.dirname(db_file)

print(f"Starting Dense GRPO Training for {active_model_cfg['name']} on [{HARDWARE_TIER}]...")
grpo_summary = run_grpo_training(
    data_path=DATA_BIRD,
    db_dir=DB_DIR,
    output_dir=LOCAL_CHECKPOINT_DIR,
    drive_dir=DRIVE_CHECKPOINT_DIR,
    num_rollouts=NUM_ROLLOUTS,
    epochs=2,
    learning_rate=5e-6,
    dry_run=True
)

print("\n=== Dense GRPO Training Summary ===")
print(json.dumps(grpo_summary, indent=2))


## Training Output & Verification
- Checkpoints saved to Google Drive: `/content/drive/MyDrive/Text2SQL_Colab/checkpoints/grpo_3b/`.
- Policy optimization successfully prevented advantage collapse through fine-grained reward heads.
- **Proceed to Notebook 04 (`notebooks/04_system1_laya421m_training.ipynb`)** for training the System-1 verifier.
